<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
import json
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


# -----------------------------
# Project paths and setup
# -----------------------------
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
PREP_ROOT = PROJECT_ROOT / "data" / "prepared"
REPORT_ROOT = PROJECT_ROOT / "reports" / "eda"
QUALITY_ROOT = PROJECT_ROOT / "data" / "quality"
QUARANTINE_ROOT = PROJECT_ROOT / "data" / "quarantine"

PREP_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
QUALITY_ROOT.mkdir(parents=True, exist_ok=True)
QUARANTINE_ROOT.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_ROOT:", RAW_ROOT)
print("PREP_ROOT:", PREP_ROOT)
print("REPORT_ROOT:", REPORT_ROOT)
print("QUALITY_ROOT:", QUALITY_ROOT)
print("QUARANTINE_ROOT:", QUARANTINE_ROOT)


# -----------------------------
# File discovery helpers
# -----------------------------
def latest_match(base_dir: Path, pattern: str):
    if not base_dir.exists():
        return None
    matches = list(base_dir.rglob(pattern))
    if not matches:
        return None
    return max(matches, key=lambda p: p.stat().st_mtime)


files = {
    "events_csv": latest_match(RAW_ROOT, "events.csv"),
    "category_tree_csv": latest_match(RAW_ROOT, "category_tree.csv"),
    "item_properties_part1_csv": latest_match(RAW_ROOT, "item_properties_part1.csv"),
    "item_properties_part2_csv": latest_match(RAW_ROOT, "item_properties_part2.csv"),
    "products_raw_json": latest_match(RAW_ROOT, "products_raw.json"),
    "categories_raw_json": latest_match(RAW_ROOT, "categories_raw.json"),
}

for name, path in files.items():
    print(f"{name}: {path}")

missing = [k for k, v in files.items() if v is None]
if missing:
    raise FileNotFoundError(f"Missing required raw files: {missing}")


# -----------------------------
# Validation helpers
# -----------------------------
def make_issue(dataset, check_type, severity, issue_count, description, file_path):
    if int(issue_count) <= 0:
        return None
    return {
        "dataset": dataset,
        "check_type": check_type,
        "severity": severity,
        "issue_count": int(issue_count),
        "description": description,
        "file_path": str(file_path),
    }


def summarize_validation(dataset, df, issues):
    errors = sum(i["issue_count"] for i in issues if i["severity"] == "error")
    warnings = sum(i["issue_count"] for i in issues if i["severity"] == "warning")
    return {
        "dataset": dataset,
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
        "errors": int(errors),
        "warnings": int(warnings),
        "status": "FAIL" if errors > 0 else "PASS",
    }


def safe_nunique(series):
    def make_hashable(x):
        if isinstance(x, (list, dict)):
            return json.dumps(x, sort_keys=True)
        return x
    return series.map(make_hashable).nunique(dropna=True)


def profile_df(df, name):
    summary = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "null_count": df.isna().sum().values,
        "null_pct": (df.isna().mean().values * 100).round(2),
        "nunique": [safe_nunique(df[col]) for col in df.columns]
    })
    print(f"\n{name} shape: {df.shape}")
    display(summary.sort_values(["null_pct", "nunique"], ascending=[False, False]))
    display(df.head())


def save_validation_reports(issues, summaries, prefix):
    issues_df = pd.DataFrame(issues)
    summary_df = pd.DataFrame(summaries)

    issues_path = QUALITY_ROOT / f"{prefix}_issues_{RUN_TS}.csv"
    summary_path = QUALITY_ROOT / f"{prefix}_summary_{RUN_TS}.csv"

    issues_df.to_csv(issues_path, index=False)
    summary_df.to_csv(summary_path, index=False)

    print(f"Saved: {issues_path}")
    print(f"Saved: {summary_path}")

    return issues_df, summary_df, issues_path, summary_path


def safe_to_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.exists() and path.is_dir():
        raise IsADirectoryError(f"Expected file but found directory: {path}")

    try:
        df.to_csv(path, index=False)
        print(f"Saved: {path}")
        return path
    except PermissionError:
        fallback = path.with_name(f"{path.stem}_{RUN_TS}{path.suffix}")
        df.to_csv(fallback, index=False)
        print(f"File locked, saved fallback instead: {fallback}")
        return fallback


# -----------------------------
# Raw validation functions
# -----------------------------
def validate_raw_events(df, file_path):
    issues = []
    d = df.copy()
    d.columns = [c.strip().lower() for c in d.columns]

    required = ["timestamp", "visitorid", "event", "itemid"]
    missing_cols = [c for c in required if c not in d.columns]
    item = make_issue("retailrocket_events", "schema", "error", len(missing_cols), f"Missing columns: {missing_cols}", file_path)
    if item:
        issues.append(item)
        return issues

    if "transactionid" not in d.columns:
        d["transactionid"] = np.nan

    item = make_issue(
        "retailrocket_events",
        "missing_values",
        "warning",
        d["transactionid"].isna().sum(),
        "Nulls in transactionid",
        file_path
    )
    if item:
        issues.append(item)

    item = make_issue(
        "retailrocket_events",
        "duplicates",
        "warning",
        d.duplicated().sum(),
        "Duplicate rows",
        file_path
    )
    if item:
        issues.append(item)

    return issues


def validate_raw_category_tree(df, file_path):
    issues = []
    d = df.copy()
    d.columns = [c.strip().lower() for c in d.columns]

    required = ["categoryid", "parentid"]
    missing_cols = [c for c in required if c not in d.columns]
    item = make_issue("retailrocket_category_tree", "schema", "error", len(missing_cols), f"Missing columns: {missing_cols}", file_path)
    if item:
        issues.append(item)
        return issues

    item = make_issue(
        "retailrocket_category_tree",
        "missing_values",
        "warning",
        d["parentid"].isna().sum(),
        "Nulls in parentid",
        file_path
    )
    if item:
        issues.append(item)

    return issues


def validate_raw_products(df, file_path):
    issues = []
    d = df.copy()
    d.columns = [c.strip().lower() for c in d.columns]

    required = ["id", "title", "category", "price"]
    missing_cols = [c for c in required if c not in d.columns]
    item = make_issue("dummyjson_products_raw", "schema", "error", len(missing_cols), f"Missing columns: {missing_cols}", file_path)
    if item:
        issues.append(item)

    if "brand" in d.columns:
        item = make_issue(
            "dummyjson_products_raw",
            "missing_values",
            "warning",
            d["brand"].isna().sum(),
            "Nulls in brand",
            file_path
        )
        if item:
            issues.append(item)

    return issues


def validate_raw_categories(df, file_path):
    issues = []
    d = df.copy()
    d.columns = [c.strip().lower() for c in d.columns]

    has_valid_category_field = any(col in d.columns for col in ["category", "slug", "name"])
    item = make_issue(
        "dummyjson_categories_raw",
        "schema",
        "error",
        0 if has_valid_category_field else 1,
        "Expected one of ['category', 'slug', 'name'] in categories payload",
        file_path
    )
    if item:
        issues.append(item)

    return issues


# -----------------------------
# Prepared validation functions
# -----------------------------
def validate_prepared_interactions(df, file_path):
    issues = []
    required = ["user_id", "item_id", "event", "event_weight", "event_ts"]
    missing_cols = [c for c in required if c not in df.columns]
    item = make_issue("interactions_prepared", "schema", "error", len(missing_cols), f"Missing columns: {missing_cols}", file_path)
    if item:
        issues.append(item)
        return issues

    item = make_issue("interactions_prepared", "duplicates", "warning", df.duplicated().sum(), "Duplicate rows", file_path)
    if item:
        issues.append(item)

    bad_txn = ((df["event"] == "transaction") & (df["transactionid"].isna())).sum()
    item = make_issue("interactions_prepared", "missing_values", "error", bad_txn, "Null transactionid for transaction events", file_path)
    if item:
        issues.append(item)

    return issues


def validate_prepared_products(df, file_path):
    issues = []
    required = ["product_id", "title", "category", "price"]
    missing_cols = [c for c in required if c not in df.columns]
    item = make_issue("products_prepared", "schema", "error", len(missing_cols), f"Missing columns: {missing_cols}", file_path)
    if item:
        issues.append(item)
        return issues

    item = make_issue("products_prepared", "duplicates", "warning", df.duplicated().sum(), "Duplicate rows", file_path)
    if item:
        issues.append(item)

    return issues


def validate_prepared_categories(df, file_path):
    issues = []
    required = ["category"]
    missing_cols = [c for c in required if c not in df.columns]
    item = make_issue("categories_prepared", "schema", "error", len(missing_cols), f"Missing columns: {missing_cols}", file_path)
    if item:
        issues.append(item)
        return issues

    blanks = (df["category"].astype(str).str.strip() == "").sum()
    item = make_issue("categories_prepared", "missing_values", "error", blanks, "Blank category values", file_path)
    if item:
        issues.append(item)

    return issues


# -----------------------------
# Load the raw ingestion files
# -----------------------------
events_raw = pd.read_csv(files["events_csv"])
category_tree_raw = pd.read_csv(files["category_tree_csv"])
item_props_1 = pd.read_csv(files["item_properties_part1_csv"])
item_props_2 = pd.read_csv(files["item_properties_part2_csv"])
item_properties_raw = pd.concat([item_props_1, item_props_2], ignore_index=True)

with open(files["products_raw_json"], "r", encoding="utf-8") as f:
    products_payload = json.load(f)

with open(files["categories_raw_json"], "r", encoding="utf-8") as f:
    categories_payload = json.load(f)

products_raw = pd.json_normalize(products_payload.get("products", []), sep="_")

if isinstance(categories_payload, list):
    if len(categories_payload) > 0 and isinstance(categories_payload[0], dict):
        categories_raw = pd.json_normalize(categories_payload, sep="_")
    else:
        categories_raw = pd.DataFrame({"category": categories_payload})
elif isinstance(categories_payload, dict):
    categories_raw = pd.json_normalize(categories_payload, sep="_")
else:
    categories_raw = pd.DataFrame()

print("events_raw:", events_raw.shape)
print("category_tree_raw:", category_tree_raw.shape)
print("item_properties_raw:", item_properties_raw.shape)
print("products_raw:", products_raw.shape)
print("categories_raw:", categories_raw.shape)


# -----------------------------
# Quick profiling snapshot
# -----------------------------
profile_df(events_raw, "events_raw")
profile_df(category_tree_raw, "category_tree_raw")
profile_df(item_properties_raw, "item_properties_raw")
profile_df(products_raw, "products_raw")
profile_df(categories_raw, "categories_raw")


# -----------------------------
# Raw validation
# -----------------------------
raw_issues = []
raw_issues.extend(validate_raw_events(events_raw, files["events_csv"]))
raw_issues.extend(validate_raw_category_tree(category_tree_raw, files["category_tree_csv"]))
raw_issues.extend(validate_raw_products(products_raw, files["products_raw_json"]))
raw_issues.extend(validate_raw_categories(categories_raw, files["categories_raw_json"]))

raw_summaries = [
    summarize_validation("retailrocket_events", events_raw, [i for i in raw_issues if i["dataset"] == "retailrocket_events"]),
    summarize_validation("retailrocket_category_tree", category_tree_raw, [i for i in raw_issues if i["dataset"] == "retailrocket_category_tree"]),
    summarize_validation("dummyjson_products_raw", products_raw, [i for i in raw_issues if i["dataset"] == "dummyjson_products_raw"]),
    summarize_validation("dummyjson_categories_raw", categories_raw, [i for i in raw_issues if i["dataset"] == "dummyjson_categories_raw"]),
]

raw_issues_df, raw_summary_df, raw_issues_path, raw_summary_path = save_validation_reports(raw_issues, raw_summaries, "raw_validation")

print("\nRAW VALIDATION ISSUES")
display(raw_issues_df)

print("\nRAW VALIDATION SUMMARY")
display(raw_summary_df)


# -----------------------------
# Clean and prepare Retailrocket interactions
# -----------------------------
events = events_raw.copy()

events.columns = [c.strip().lower() for c in events.columns]

required_event_cols = ["timestamp", "visitorid", "event", "itemid"]
missing_cols = [c for c in required_event_cols if c not in events.columns]
if missing_cols:
    raise ValueError(f"events.csv missing columns: {missing_cols}")

events = events.dropna(subset=["timestamp", "visitorid", "event", "itemid"]).copy()

events["timestamp"] = pd.to_numeric(events["timestamp"], errors="coerce")
events["visitorid"] = pd.to_numeric(events["visitorid"], errors="coerce")
events["itemid"] = pd.to_numeric(events["itemid"], errors="coerce")

events = events.dropna(subset=["timestamp", "visitorid", "itemid"]).copy()

events["event"] = events["event"].astype(str).str.strip().str.lower()
events = events[events["event"].isin(["view", "addtocart", "transaction"])].copy()

if "transactionid" not in events.columns:
    events["transactionid"] = np.nan

events["transactionid"] = pd.to_numeric(events["transactionid"], errors="coerce")
events["transactionid_missing"] = events["transactionid"].isna().astype(int)

bad_transactions = events[(events["event"] == "transaction") & (events["transactionid"].isna())].copy()
if not bad_transactions.empty:
    bad_transactions.to_csv(QUARANTINE_ROOT / f"invalid_transaction_rows_{RUN_TS}.csv", index=False)

events = events[~((events["event"] == "transaction") & (events["transactionid"].isna()))].copy()

events["event_ts"] = pd.to_datetime(events["timestamp"], unit="ms", errors="coerce")
events = events.dropna(subset=["event_ts"]).copy()

events["user_id"] = events["visitorid"].astype("int64")
events["item_id"] = events["itemid"].astype("int64")
events["event_date"] = events["event_ts"].dt.date
events["event_hour"] = events["event_ts"].dt.hour
events["event_dow"] = events["event_ts"].dt.day_name()

event_weight_map = {"view": 1.0, "addtocart": 3.0, "transaction": 5.0}
events["event_weight"] = events["event"].map(event_weight_map).astype(float)

events = events.drop_duplicates(subset=["timestamp", "user_id", "item_id", "event"]).copy()

interactions_prepared = events[
    [
        "user_id",
        "item_id",
        "event",
        "event_weight",
        "event_ts",
        "event_date",
        "event_hour",
        "event_dow",
        "transactionid",
        "transactionid_missing",
    ]
].reset_index(drop=True)

print(interactions_prepared.shape)
display(interactions_prepared.head())


# -----------------------------
# Prepare Retailrocket item attributes snapshot
# -----------------------------
category_tree = category_tree_raw.copy()
category_tree.columns = [c.strip().lower() for c in category_tree.columns]

if "parentid" in category_tree.columns:
    category_tree["is_root"] = category_tree["parentid"].isna().astype(int)
    category_tree["parentid"] = category_tree["parentid"].fillna(-1)

item_properties = item_properties_raw.copy()
item_properties.columns = [c.strip().lower() for c in item_properties.columns]

item_properties = item_properties.dropna(subset=["timestamp", "itemid", "property", "value"]).copy()
item_properties["timestamp"] = pd.to_numeric(item_properties["timestamp"], errors="coerce")
item_properties["itemid"] = pd.to_numeric(item_properties["itemid"], errors="coerce")
item_properties["prop_ts"] = pd.to_datetime(item_properties["timestamp"], unit="ms", errors="coerce")
item_properties = item_properties.dropna(subset=["timestamp", "itemid", "prop_ts"]).copy()

item_properties["item_id"] = item_properties["itemid"].astype("int64")
item_properties["property"] = item_properties["property"].astype(str).str.strip().str.lower()
item_properties["value"] = item_properties["value"].astype(str).str.strip()

latest_item_props = (
    item_properties
    .sort_values(["item_id", "property", "prop_ts"])
    .drop_duplicates(subset=["item_id", "property"], keep="last")
)

top_properties = (
    latest_item_props["property"]
    .value_counts()
    .head(25)
    .index
    .tolist()
)

item_snapshot = (
    latest_item_props[latest_item_props["property"].isin(top_properties)]
    .pivot(index="item_id", columns="property", values="value")
    .reset_index()
)

print("Top properties used:", len(top_properties))
print("item_snapshot:", item_snapshot.shape)
display(item_snapshot.head())


# -----------------------------
# Clean, encode, and normalize DummyJSON products
# -----------------------------
def minmax_scale(series):
    s = pd.to_numeric(series, errors="coerce")
    s_min = s.min()
    s_max = s.max()
    if pd.isna(s_min) or pd.isna(s_max) or s_min == s_max:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s_min) / (s_max - s_min)


products = products_raw.copy()
products.columns = [c.strip().lower() for c in products.columns]

required_product_cols = ["id", "title", "category", "price"]
missing_product_cols = [c for c in required_product_cols if c not in products.columns]
if missing_product_cols:
    raise ValueError(f"products_raw.json missing fields: {missing_product_cols}")

products = products.drop_duplicates(subset=["id"]).copy()
products["id"] = pd.to_numeric(products["id"], errors="coerce")
products["price"] = pd.to_numeric(products.get("price"), errors="coerce")
products["rating"] = pd.to_numeric(products.get("rating"), errors="coerce")
products["stock"] = pd.to_numeric(products.get("stock"), errors="coerce")
products["discountpercentage"] = pd.to_numeric(products.get("discountpercentage"), errors="coerce")

products["title"] = products["title"].astype(str).str.strip()
products["category"] = products["category"].astype(str).str.strip().str.lower()

if "brand" not in products.columns:
    products["brand"] = "Unknown"
else:
    products["brand"] = products["brand"].fillna("Unknown").astype(str).str.strip()
    products.loc[products["brand"].eq(""), "brand"] = "Unknown"

products = products.dropna(subset=["id", "title", "category", "price"]).copy()
products = products[products["price"] >= 0].copy()

products["product_id"] = products["id"].astype("int64")
products["price_norm"] = minmax_scale(products["price"])
products["rating_norm"] = minmax_scale(products["rating"])
products["stock_norm"] = minmax_scale(products["stock"])
products["discount_norm"] = minmax_scale(products["discountpercentage"])

category_dummies = pd.get_dummies(products["category"], prefix="cat", dtype=int)
brand_dummies = pd.get_dummies(products["brand"], prefix="brand", dtype=int)

products_prepared = pd.concat(
    [
        products[
            [
                "product_id",
                "title",
                "category",
                "brand",
                "price",
                "rating",
                "stock",
                "discountpercentage",
                "price_norm",
                "rating_norm",
                "stock_norm",
                "discount_norm",
            ]
        ].reset_index(drop=True),
        category_dummies.reset_index(drop=True),
        brand_dummies.reset_index(drop=True),
    ],
    axis=1,
)

print(products_prepared.shape)
display(products_prepared.head())


# -----------------------------
# Normalize categories raw
# -----------------------------
if isinstance(categories_payload, list):
    if len(categories_payload) > 0 and isinstance(categories_payload[0], dict):
        categories_prepared = pd.json_normalize(categories_payload, sep="_")

        if "slug" in categories_prepared.columns:
            categories_prepared = categories_prepared[["slug"]].rename(columns={"slug": "category"})
        elif "name" in categories_prepared.columns:
            categories_prepared = categories_prepared[["name"]].rename(columns={"name": "category"})
        elif "category" in categories_prepared.columns:
            categories_prepared = categories_prepared[["category"]]
        else:
            raise ValueError("categories payload does not contain slug, name, or category")
    else:
        categories_prepared = pd.DataFrame({"category": categories_payload})

elif isinstance(categories_payload, dict) and "categories" in categories_payload:
    categories_prepared = pd.DataFrame({"category": categories_payload["categories"]})

else:
    categories_prepared = categories_raw.copy()

    if "slug" in categories_prepared.columns:
        categories_prepared = categories_prepared[["slug"]].rename(columns={"slug": "category"})
    elif "name" in categories_prepared.columns:
        categories_prepared = categories_prepared[["name"]].rename(columns={"name": "category"})
    elif "category" in categories_prepared.columns:
        categories_prepared = categories_prepared[["category"]]
    else:
        raise ValueError("categories_raw does not contain slug, name, or category")

categories_prepared["category"] = (
    categories_prepared["category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

categories_prepared = (
    categories_prepared[categories_prepared["category"] != ""]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("categories_prepared:", categories_prepared.shape)
display(categories_prepared.head())


# -----------------------------
# Rerun validation on prepared outputs
# -----------------------------
prepared_issues = []
prepared_issues.extend(validate_prepared_interactions(interactions_prepared, PREP_ROOT / "interactions_prepared.csv"))
prepared_issues.extend(validate_prepared_products(products_prepared, PREP_ROOT / "products_prepared.csv"))
prepared_issues.extend(validate_prepared_categories(categories_prepared, PREP_ROOT / "categories_prepared.csv"))

prepared_summaries = [
    summarize_validation("interactions_prepared", interactions_prepared, [i for i in prepared_issues if i["dataset"] == "interactions_prepared"]),
    summarize_validation("products_prepared", products_prepared, [i for i in prepared_issues if i["dataset"] == "products_prepared"]),
    summarize_validation("categories_prepared", categories_prepared, [i for i in prepared_issues if i["dataset"] == "categories_prepared"]),
]

prepared_issues_df, prepared_summary_df, prepared_issues_path, prepared_summary_path = save_validation_reports(prepared_issues, prepared_summaries, "prepared_validation")

print("\nPREPARED VALIDATION ISSUES")
display(prepared_issues_df)

print("\nPREPARED VALIDATION SUMMARY")
display(prepared_summary_df)


# -----------------------------
# EDA: interaction distributions and popularity
# -----------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.countplot(data=interactions_prepared, x="event", order=["view", "addtocart", "transaction"], ax=axes[0, 0])
axes[0, 0].set_title("Interaction Type Distribution")
axes[0, 0].set_xlabel("Event Type")
axes[0, 0].set_ylabel("Count")

daily_interactions = interactions_prepared.groupby("event_date").size().reset_index(name="interaction_count")
sns.lineplot(data=daily_interactions, x="event_date", y="interaction_count", ax=axes[0, 1])
axes[0, 1].set_title("Daily Interaction Volume")
axes[0, 1].set_xlabel("Date")
axes[0, 1].set_ylabel("Interactions")
axes[0, 1].tick_params(axis="x", rotation=45)

top_items = (
    interactions_prepared.groupby("item_id")
    .size()
    .sort_values(ascending=False)
    .head(20)
    .reset_index(name="interaction_count")
)
sns.barplot(data=top_items, x="item_id", y="interaction_count", ax=axes[1, 0], color="#86BC25")
axes[1, 0].set_title("Top 20 Most Popular Items")
axes[1, 0].set_xlabel("Item ID")
axes[1, 0].set_ylabel("Interactions")
axes[1, 0].tick_params(axis="x", rotation=90)

user_activity = (
    interactions_prepared.groupby("user_id")
    .size()
    .reset_index(name="interaction_count")
)
sns.histplot(user_activity["interaction_count"], bins=40, ax=axes[1, 1], color="#0076A8")
axes[1, 1].set_title("User Activity Distribution")
axes[1, 1].set_xlabel("Interactions per User")
axes[1, 1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


# -----------------------------
# EDA: product distributions
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(products_prepared["price"], bins=30, ax=axes[0], color="#86BC25")
axes[0].set_title("Product Price Distribution")
axes[0].set_xlabel("Price")
axes[0].set_ylabel("Count")

sns.histplot(products_prepared["rating"].dropna(), bins=20, ax=axes[1], color="#0076A8")
axes[1].set_title("Product Rating Distribution")
axes[1].set_xlabel("Rating")
axes[1].set_ylabel("Count")

top_categories = (
    products_prepared["category"]
    .value_counts()
    .head(15)
    .reset_index()
)
top_categories.columns = ["category", "count"]
sns.barplot(data=top_categories, x="category", y="count", ax=axes[2], color="#046E38")
axes[2].set_title("Top Product Categories")
axes[2].set_xlabel("Category")
axes[2].set_ylabel("Count")
axes[2].tick_params(axis="x", rotation=60)

plt.tight_layout()
plt.show()


# -----------------------------
# EDA: sparsity pattern heatmap
# -----------------------------
top_users = (
    interactions_prepared.groupby("user_id")
    .size()
    .sort_values(ascending=False)
    .head(50)
    .index
)

top_items = (
    interactions_prepared.groupby("item_id")
    .size()
    .sort_values(ascending=False)
    .head(50)
    .index
)

sample_matrix = (
    interactions_prepared[
        interactions_prepared["user_id"].isin(top_users) &
        interactions_prepared["item_id"].isin(top_items)
    ]
    .pivot_table(
        index="user_id",
        columns="item_id",
        values="event_weight",
        aggfunc="sum",
        fill_value=0
    )
)

if sample_matrix.size > 0:
    sparsity = 1 - (np.count_nonzero(sample_matrix.values) / sample_matrix.size)
else:
    sparsity = np.nan

plt.figure(figsize=(14, 8))
sns.heatmap(sample_matrix > 0, cmap="Blues", cbar=False)
plt.title(f"User-Item Interaction Sparsity Heatmap (sample) | sparsity={sparsity:.2%}" if pd.notna(sparsity) else "User-Item Interaction Sparsity Heatmap (sample)")
plt.xlabel("Item ID")
plt.ylabel("User ID")
plt.show()

print("Matrix shape:", sample_matrix.shape)
print("Sample sparsity:", round(sparsity, 4) if pd.notna(sparsity) else "N/A")


# -----------------------------
# Final prepared outputs and summary report
# -----------------------------
interactions_path = safe_to_csv(interactions_prepared, PREP_ROOT / "interactions_prepared.csv")
item_snapshot_path = safe_to_csv(item_snapshot, PREP_ROOT / "retailrocket_item_snapshot.csv")
products_path = safe_to_csv(products_prepared, PREP_ROOT / "products_prepared.csv")
categories_path = safe_to_csv(categories_prepared, PREP_ROOT / "categories_prepared.csv")

eda_summary = {
    "run_ts": RUN_TS,
    "project_root": str(PROJECT_ROOT),
    "raw_files": {k: str(v) for k, v in files.items()},
    "prepared_outputs": {
        "interactions_prepared": str(interactions_path),
        "item_snapshot": str(item_snapshot_path),
        "products_prepared": str(products_path),
        "categories_prepared": str(categories_path),
        "raw_validation_issues": str(raw_issues_path),
        "raw_validation_summary": str(raw_summary_path),
        "prepared_validation_issues": str(prepared_issues_path),
        "prepared_validation_summary": str(prepared_summary_path),
        "eda_summary_json": str(REPORT_ROOT / f"eda_summary_{RUN_TS}.json"),
    },
    "metrics": {
        "interaction_rows": int(len(interactions_prepared)),
        "unique_users": int(interactions_prepared["user_id"].nunique()),
        "unique_items": int(interactions_prepared["item_id"].nunique()),
        "product_rows": int(len(products_prepared)),
        "category_rows": int(len(categories_prepared)),
        "unique_categories": int(products_prepared["category"].nunique()),
        "sample_sparsity": None if pd.isna(sparsity) else float(sparsity),
        "quarantined_bad_transaction_rows": int(len(bad_transactions)),
    }
}

with open(REPORT_ROOT / f"eda_summary_{RUN_TS}.json", "w", encoding="utf-8") as f:
    json.dump(eda_summary, f, indent=2)

print("Saved prepared files:")
for k, v in eda_summary["prepared_outputs"].items():
    print(f"- {k}: {v}")

display(pd.DataFrame([eda_summary["metrics"]]))


c:\Users\barath\AppData\Local\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


ModuleNotFoundError: No module named 'matplotlib.backends.registry'